In [ ]:
def rollout_from_env(
    env,
    instruction: str,
    horizon: int,
    policy_tokenizer,
    policy_model,
    critic,
    max_new_tokens: int = 64,
):
    """Collect a single real-environment episode with the current policy model."""
    obs = env.reset({"instruction": instruction})
    history: List[Dict[str, Any]] = []
    steps: List[Dict[str, Any]] = []

    for _t in range(horizon):
        prompt = build_user_prompt(instruction, obs, history)
        inputs = policy_tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You are a web agent."},
                {"role": "user", "content": prompt},
            ],
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(policy_model.pretrained_model.device)
        out = policy_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            eos_token_id=policy_tokenizer.eos_token_id,
        )
        raw = policy_tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        action = _extract_action(raw)

        obs_next, done, _info = env.step(action)
        reward = 0.0
        if critic is not None:
            reward = float(critic.score(instruction, [(obs, action, obs_next)]))

        steps.append({"prompt": prompt, "action": action, "next_obs": obs_next, "reward": reward})
        history.append({"action": action, "observation": obs_next})
        obs = obs_next
        if done:
            break

    return steps

